# MPMWorlds 数据与参数分布分析

目标：在设计 split 和 SFT 之前，先用数据回答四个问题。

1. **数据长什么样**：schema 覆盖率、物体组成、材料分布
2. **参数是不是真的连续分布**：还是少数几个"整数值"（LLM 生成的程序常见）
3. **yaml 和 program.py 一致吗**：yaml 是不是代码的有损摘要
4. **split 有没有泄漏**：模板重复、近重复程序、结构重叠

用法：只需改下面的配置 cell，然后 Run All。每个 section 末尾的 `# >>> 看什么` 注释说明该关注哪些输出。

In [ ]:
from pathlib import Path
import re, json, hashlib, collections, math, random
import numpy as np, pandas as pd
import yaml
import matplotlib.pyplot as plt

# ======================= 配置：改这里 =======================
DATA_ROOT  = Path("/path/to/mpmworlds")       # 根目录；递归查找包含 YAML_NAME 的目录，每个目录视为一个 scene
YAML_NAME  = "scene.yaml"
PROG_NAME  = "program.py"
VIDEO_NAME = "video.mp4"

# 你已有的划分（可选）。每个文件一行一个 scene 标识（相对 DATA_ROOT 的路径，或目录名）
SPLIT_FILES = {}                               # 例: {"train": "train.txt", "val": "val.txt", "test": "test.txt"}

TOKENIZER   = None                             # 例: "Qwen/Qwen2.5-VL-3B-Instruct"；None 则用 字符数/3.5 粗估
CHECK_VIDEO = True
VIDEO_SAMPLE = 300                             # 最多抽查多少个视频
BOILER_DF   = 0.5                              # 出现在 >=50% 程序里的行视为样板
OUT_DIR = Path("analysis_out"); OUT_DIR.mkdir(exist_ok=True)
# ============================================================

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})

def safe_hist(s, bins=40, title="", ax=None):
    # 常量/低基数数据时 np.histogram 会报错，退化为取值计数的柱状图
    s = pd.Series(s).dropna()
    ax = ax or plt.gca()
    if s.nunique() <= 1 or s.max() - s.min() < 1e-9:
        s.value_counts().sort_index().plot.bar(ax=ax)
    else:
        ax.hist(s, bins=min(bins, max(5, s.nunique())))
    ax.set_title(title)
pd.set_option("display.width", 200, "display.max_columns", 50, "display.max_colwidth", 80)

## 1. 载入与完整性检查

In [ ]:
scene_dirs = sorted({p.parent for p in DATA_ROOT.rglob(YAML_NAME)})
print(f"found {len(scene_dirs)} scene dirs under {DATA_ROOT}")
assert scene_dirs, "没找到 scene.yaml：检查 DATA_ROOT / YAML_NAME"

def to_num(x):
    # 注意：PyYAML 把 "5e3"（无小数点）解析成字符串，必须手动转
    if isinstance(x, bool): return np.nan
    if isinstance(x, (int, float)): return float(x)
    if isinstance(x, str):
        try: return float(x)
        except ValueError: return np.nan
    return np.nan

specs, ytext, progs, problems = {}, {}, {}, []
for d in scene_dirs:
    sid = str(d.relative_to(DATA_ROOT))
    yp, pp, vp = d / YAML_NAME, d / PROG_NAME, d / VIDEO_NAME
    try:
        txt = yp.read_text()
        spec = yaml.safe_load(txt)
        assert isinstance(spec, dict)
    except Exception as e:
        problems.append((sid, "yaml_parse", str(e)[:100])); continue
    specs[sid], ytext[sid] = spec, txt
    if pp.exists(): progs[sid] = pp.read_text(errors="ignore")
    else: problems.append((sid, "missing_program", ""))
    if not vp.exists(): problems.append((sid, "missing_video", ""))

print(len(specs), "yaml ok |", len(progs), "programs |", len(problems), "problems")
if problems:
    display(pd.DataFrame(problems, columns=["scene", "type", "detail"]).groupby("type").size().rename("count").to_frame())

## 2. Schema 覆盖率与物体组成

In [ ]:
def flat(prefix, d, out):
    for k, v in (d or {}).items():
        if isinstance(v, dict): flat(f"{prefix}{k}.", v, out)
        else: out[f"{prefix}{k}"] = v
    return out

scene_rows, obj_rows = [], []
for sid, spec in specs.items():
    objs = spec.get("objects") or []
    sc = flat("", {k: v for k, v in spec.items() if k != "objects"}, {})
    sc["scene"], sc["n_objects"] = sid, len(objs)
    scene_rows.append(sc)
    for i, o in enumerate(objs):
        r = flat("", o, {}); r.update(scene=sid, obj_idx=i); obj_rows.append(r)
scene_df, obj_df = pd.DataFrame(scene_rows), pd.DataFrame(obj_rows)

def col(df, *names):
    for n in names:
        if n in df: return df[n]
    return pd.Series([np.nan] * len(df), index=df.index)

obj_df["otype"] = col(obj_df, "object_type")
obj_df["shape"] = col(obj_df, "shape.type", "initial_state.shape.type")
obj_df["cm"]    = col(obj_df, "physics_properties.constitutive_model")
obj_df["em"]    = col(obj_df, "physics_properties.elasticity_model")
for k, n in {"rho": "density", "E": "youngs_modulus", "nu": "poissons_ratio", "fric": "friction_coefficient"}.items():
    obj_df[k] = col(obj_df, f"physics_properties.{n}").map(to_num)

# --- key path 覆盖率（按 scene 计）---
def paths(x, pre="", out=None):
    out = set() if out is None else out
    if isinstance(x, dict):
        for k, v in x.items(): paths(v, f"{pre}.{k}" if pre else str(k), out)
    elif isinstance(x, list):
        if all(not isinstance(i, (dict, list)) for i in x): out.add(pre)
        else:
            for i in x: paths(i, pre + "[]", out)
    else: out.add(pre)
    return out

cnt = collections.Counter()
for spec in specs.values(): cnt.update(paths(spec))
cov = pd.Series(cnt).div(len(specs)).sort_values(ascending=False).rename("share_of_scenes")
display(cov.to_frame().style.format("{:.1%}"))
# >>> 看什么：份额在 5%~95% 之间的字段 = 可选字段；这些字段的有无本身就是要预测的东西，schema 设计要条件化

In [ ]:
print("object_type:");       display(obj_df.otype.value_counts(dropna=False).to_frame())
print("constitutive_model:"); display(obj_df.cm.value_counts(dropna=False).to_frame())
print("cm x em:");            display(pd.crosstab(obj_df.cm.fillna("-"), obj_df.em.fillna("-")))
print("shape.type:");         display(obj_df["shape"].value_counts(dropna=False).to_frame())
print("n_objects per scene:");display(scene_df.n_objects.value_counts().sort_index().to_frame())

# --- 结构签名：(object_type, shape, constitutive_model) 的多重集合 ---
def obj_sig(o):
    pp = o.get("physics_properties") or {}
    shp = o.get("shape") or (o.get("initial_state") or {}).get("shape") or {}
    return str((o.get("object_type"), shp.get("type"), pp.get("constitutive_model")))
def scene_sig(spec): return " | ".join(sorted(obj_sig(o) for o in spec.get("objects") or []))
scene_df["sig"] = scene_df.scene.map(lambda s: scene_sig(specs[s]))
vc = scene_df.sig.value_counts()
print(f"unique structure signatures: {len(vc)} / {len(scene_df)} scenes")
print(f"top1 / top5 / top10 share: {vc.iloc[:1].sum()/len(scene_df):.1%} / {vc.iloc[:5].sum()/len(scene_df):.1%} / {vc.iloc[:10].sum()/len(scene_df):.1%}")
display(vc.head(15).to_frame())
# >>> 看什么：top1 占比过高 => 数据多样性不足；签名数很少 => 按签名分组划分会让某些组极大，需要分层处理

## 3. 物理参数分布

关注三件事：**每种材料的样本量**、**参数是连续还是离散（整数值）**、**ρ/E/ν 之间是否相关**。
离散度尤其重要：如果 E 只取 {1e3, 5e3, 1e4, 1e5} 这几个值，那它实际是分类问题，不是回归问题，评估和损失设计都不同。

In [ ]:
dyn = obj_df[obj_df.cm.notna()].copy()           # 有本构模型的物体
print("dynamic-like objects:", len(dyn), "| scenes:", dyn.scene.nunique())

def sig2(x, n=2):                                  # 是否 <=n 位有效数字
    if not np.isfinite(x) or x == 0: return False
    return float(f"{x:.{n-1}e}") == x

def param_table(df, params, by):
    out = []
    for g, sub in df.groupby(by):
        for p in params:
            s = sub[p].dropna()
            if s.empty: continue
            vc = s.round(12).value_counts()
            out.append(dict(group=g, param=p, n=len(s), n_unique=s.nunique(),
                            top_value=vc.index[0], top_share=vc.iloc[0] / len(s),
                            round_share=s.map(sig2).mean(),
                            min=s.min(), median=s.median(), max=s.max(),
                            log10_span=(np.log10(s.max() / s.min()) if s.min() > 0 else np.nan)))
    return pd.DataFrame(out)

pt = param_table(dyn, ["rho", "E", "nu"], "cm")
display(pt.style.format({"top_share": "{:.1%}", "round_share": "{:.1%}", "min": "{:.3g}", "median": "{:.3g}", "max": "{:.3g}", "log10_span": "{:.2f}"}))
# >>> 看什么：
#   n_unique 很小 / top_share 很高 / round_share≈100%  => 参数是 LLM 挑的"整数值"，不是连续采样
#   log10_span 大 => 必须用 log 空间；n 很小的材料 => 该材料的参数指标不可靠，单独报告

In [ ]:
def hist_grid(df, specs_, by, max_groups=6, bins=30):
    groups = df[by].value_counts().index[:max_groups]
    fig, axes = plt.subplots(len(groups), len(specs_), figsize=(4 * len(specs_), 2.4 * len(groups)), squeeze=False)
    for i, g in enumerate(groups):
        for j, (p, f, label) in enumerate(specs_):
            s = df.loc[df[by] == g, p].dropna()
            s = s[s > 0] if f is np.log10 else s
            ax = axes[i, j]
            if len(s): ax.hist(f(s), bins=bins)
            ax.set_title(f"{g} | {label} (n={len(s)})", fontsize=8)
    plt.tight_layout(); plt.show()

hist_grid(dyn, [("rho", np.log10, "log10 ρ"), ("E", np.log10, "log10 E"), ("nu", lambda x: x, "ν")], "cm")

In [ ]:
# 摩擦系数：在示例里挂在 collider 上，所以按 object_type 看
fr = obj_df[obj_df.fric.notna()]
if len(fr):
    print(param_table(fr, ["fric"], "otype").to_string())
    fig, axs = plt.subplots(1, fr.otype.nunique(), figsize=(4 * fr.otype.nunique(), 3), squeeze=False)
    for ax, (g, sub) in zip(axs[0], fr.groupby("otype")): safe_hist(sub.fric, 30, f"friction | {g}", ax=ax)
    plt.tight_layout(); plt.show()
else:
    print("no friction_coefficient found under physics_properties")

In [ ]:
# 派生量：这些才是动力学真正依赖的组合
dyn["lam"] = dyn.E * dyn.nu / ((1 + dyn.nu) * (1 - 2 * dyn.nu))
dyn["mu"]  = dyn.E / (2 * (1 + dyn.nu))
dyn["E_over_rho"] = dyn.E / dyn.rho
dyn["lam_over_rho"] = dyn.lam / dyn.rho
dyn["c_wave"] = np.sqrt(dyn.E_over_rho)

# 相关性（Spearman，log 空间）
lg = pd.DataFrame({"log_rho": np.log10(dyn.rho), "log_E": np.log10(dyn.E), "nu": dyn.nu, "cm": dyn.cm})
for g, sub in lg.groupby("cm"):
    if len(sub) >= 10:
        print(f"--- {g} (n={len(sub)})"); display(sub.drop(columns="cm").corr(method="spearman").round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for g, sub in dyn.groupby("cm"):
    ax[0].scatter(np.log10(sub.rho), np.log10(sub.E), s=10, alpha=.6, label=g)
    ax[1].hist(np.log10(sub.E_over_rho.dropna()), bins=30, alpha=.5, label=g)
ax[0].set_xlabel("log10 ρ"); ax[0].set_ylabel("log10 E"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("log10 (E/ρ)"); ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()
# >>> 看什么：ρ 与 E 强相关 => 材料先验已经编码在数据里（模型可能靠"材料→典型参数"而非读视频）；
#     E/ρ 的离散程度 = 真正可辨识量的有效取值数

## 4. 仿真配置与时间尺度

关键点：视频帧率不等于物理时间。每帧对应的仿真时间 `substeps × dt` 如果在 scene 之间变化，它是一个**视频里读不出来的潜变量**（只能从运动幅度间接推），必须决定它是固定常量还是要预测。

In [ ]:
sc = scene_df.copy()
sc["grid_res"] = col(sc, "simulation.grid_resolution").map(to_num)
sc["dt"]       = col(sc, "simulation.time_step_dt").map(to_num)
sc["substeps"] = col(sc, "simulation.substeps_per_frame").map(to_num)
sc["frames"]   = col(sc, "simulation.duration_frames").map(to_num)
sc["fps"]      = col(sc, "output.video_framerate").map(to_num)
sc["dim"]      = col(sc, "simulation.dimension").map(to_num)
sc["sim_dt_per_frame"] = sc.substeps * sc.dt
sc["sim_time_total"]   = sc.sim_dt_per_frame * sc.frames
sc["sim_speed"]        = sc.sim_dt_per_frame * sc.fps           # 1 秒视频对应多少秒仿真时间
sc["gravity"] = col(sc, "environment.gravity").astype(str)

for c in ["dim", "grid_res", "dt", "substeps", "frames", "fps", "sim_dt_per_frame", "sim_time_total", "sim_speed", "gravity"]:
    s = sc[c].dropna()
    vc = s.value_counts()
    print(f"{c:18s} n_unique={s.nunique():4d} | top: " + ", ".join(f"{k}({v})" for k, v in vc.head(4).items()))

# CFL 型数：dt * sqrt(E/rho) / dx，>~1 时显式 MPM 容易炸
d2 = dyn.merge(sc[["scene", "grid_res", "dt"]], on="scene", how="left")
d2["cfl"] = d2.dt * np.sqrt(d2.E / d2.rho) * d2.grid_res
print("\nCFL-like number by cm:")
display(d2.groupby("cm").cfl.describe().round(3))
# >>> 看什么：sim_dt_per_frame 的 n_unique。=1 => 时间尺度固定，可以写进 prompt；>1 => 必须作为预测目标或作为已知条件

## 5. 程序统计：长度、样板占比、模板重复、近重复

这一节直接决定 SFT 设计：
- 样板占比高 => code 的 token 损失被稀释，需要分段/加权损失，或改成"生成 scene + 确定性编译"
- 模板/近重复多 => 随机 split 必然泄漏

In [ ]:
if TOKENIZER:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(TOKENIZER)
    ntok = lambda s: len(tok(s, add_special_tokens=False).input_ids)
    print("tokenizer:", TOKENIZER)
else:
    ntok = lambda s: int(len(s) / 3.5)
    print("tokenizer 未设置：token 数为字符数/3.5 的粗估")

ps = pd.DataFrame({"scene": list(progs)})
ps["code_chars"]  = ps.scene.map(lambda s: len(progs[s]))
ps["code_lines"]  = ps.scene.map(lambda s: progs[s].count("\n") + 1)
ps["code_tokens"] = ps.scene.map(lambda s: ntok(progs[s]))
ps["yaml_tokens"] = ps.scene.map(lambda s: ntok(ytext[s]))
ps["ratio_yaml_to_code"] = ps.yaml_tokens / ps.code_tokens
display(ps[["code_chars", "code_lines", "code_tokens", "yaml_tokens", "ratio_yaml_to_code"]].describe(percentiles=[.5, .9, .99]).round(2))
for t in (2048, 4096, 8192):
    print(f"code_tokens > {t}: {(ps.code_tokens > t).mean():.1%}")
safe_hist(ps.code_tokens, 40, "code tokens"); plt.show()
# >>> 看什么：code 长度的 P99 决定 max_len；加上视频 token 后是否超显存；yaml/code 比值=scene 目标相对 code 目标的"信息密度"

In [ ]:
# --- 样板行占比 ---
def norm_line(l): return re.sub(r"\s+", " ", l.split("#")[0]).strip()
plines, df_cnt = {}, collections.Counter()
for sid, c in progs.items():
    ls = [x for x in (norm_line(l) for l in c.splitlines()) if x]
    plines[sid] = ls; df_cnt.update(set(ls))
N = len(progs)
boiler = {l for l, c in df_cnt.items() if c / N >= BOILER_DF}
ps["boiler_frac"] = ps.scene.map(lambda s: sum(len(l) for l in plines[s] if l in boiler) / max(1, sum(len(l) for l in plines[s])))
num_re = re.compile(r"\d")
ps["nonboiler_numeric_lines"] = ps.scene.map(lambda s: sum(1 for l in plines[s] if l not in boiler and num_re.search(l)))
print(f"样板行集合大小: {len(boiler)} 行（出现在 >= {BOILER_DF:.0%} 的程序中）")
print(ps[["boiler_frac", "nonboiler_numeric_lines"]].describe().round(3))
# >>> 看什么：boiler_frac 的均值。例如 0.7 => 70% 的 code 字符是所有程序共有的，token 平均 CE 会被它主导

In [ ]:
# --- 模板哈希（数字、注释、空白归一化后完全相同）---
def template(code):
    c = re.sub(r"#.*", "", code)
    c = re.sub(r"\b\d+\.?\d*(?:[eE][-+]?\d+)?\b", "N", c)
    return hashlib.md5(re.sub(r"\s+", " ", c).encode()).hexdigest()[:12]
ps["tmpl"] = ps.scene.map(lambda s: template(progs[s]))
g = ps.tmpl.value_counts()
print(f"unique templates: {len(g)} / {len(ps)} | scenes in templates of size>1: {(ps.tmpl.map(g) > 1).mean():.1%} | largest template: {g.iloc[0]}")

# --- 近重复：TF-IDF(标识符 1-3gram) 余弦最近邻 ---
from sklearn.feature_extraction.text import TfidfVectorizer
ids = list(progs)
clean = [re.sub(r"#.*", "", progs[i]) for i in ids]
vec = TfidfVectorizer(token_pattern=r"[A-Za-z_][A-Za-z_0-9]*", ngram_range=(1, 3), min_df=2, max_features=100000, sublinear_tf=True)
X = vec.fit_transform(clean)
nn_sim, nn_idx = np.zeros(len(ids)), np.zeros(len(ids), int)
for s in range(0, len(ids), 500):
    S = (X[s:s+500] @ X.T).toarray()
    for r in range(S.shape[0]): S[r, s + r] = -1
    nn_idx[s:s+500], nn_sim[s:s+500] = S.argmax(1), S.max(1)
ps["nn_sim"] = nn_sim; ps["nn_scene"] = [ids[j] for j in nn_idx]
print("nearest-neighbour cosine similarity quantiles:"); print(ps.nn_sim.describe(percentiles=[.1, .5, .9, .99]).round(3))
for t in (.9, .95, .99): print(f"nn_sim > {t}: {(ps.nn_sim > t).mean():.1%}")
safe_hist(ps.nn_sim, 40, "max cosine similarity to any other program"); plt.show()
# >>> 看什么：nn_sim>0.95 的比例 = 随机 split 下"测试程序在训练集里有近似拷贝"的概率上界

## 6. yaml ↔ program.py 一致性

yaml 看起来像是从代码总结出来的（带自然语言 description）。如果 yaml 与代码不一致，用 yaml 当 ground truth 就是在训练一个**与仿真行为不符的标签**。这里用正则抽取代码里的常见赋值与 yaml 对比（尽力而为：变量名不规范的程序会记为"未抽取"）。

In [ ]:
NUM = r"([-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?\d+)?)"
def first_num(pats, code):
    for p in pats:
        m = re.search(p, code, re.M)
        if m: return [to_num(g) for g in m.groups()]
    return None

sc_i = sc.set_index("scene")
first_dyn = dyn.sort_values(["scene", "obj_idx"]).groupby("scene").first()
checks = {  # name: (code patterns, yaml getter)
    "grid_res":   ([rf"^\s*grid_res\s*=\s*{NUM}"],      lambda s: sc_i.at[s, "grid_res"]),
    "dt":         ([rf"^\s*dt\s*=\s*{NUM}"],            lambda s: sc_i.at[s, "dt"]),
    "substeps":   ([rf"^\s*substeps\s*=\s*{NUM}"],      lambda s: sc_i.at[s, "substeps"]),
    "frames":     ([rf"^\s*max_frames\s*=\s*{NUM}"],    lambda s: sc_i.at[s, "frames"]),
    "rho":        ([rf"^\s*rho\s*=\s*{NUM}"],           lambda s: first_dyn.rho.get(s, np.nan)),
    "E":          ([rf"^\s*E\s*,\s*nu\s*=\s*{NUM}\s*,", rf"^\s*E\s*=\s*{NUM}"], lambda s: first_dyn.E.get(s, np.nan)),
    "nu":         ([rf"^\s*E\s*,\s*nu\s*=\s*[^,\n]+,\s*{NUM}", rf"^\s*nu\s*=\s*{NUM}"], lambda s: first_dyn.nu.get(s, np.nan)),
}
res, mism = [], []
for name, (pats, getter) in checks.items():
    ok = bad = miss = 0
    for s in progs:
        got = first_num(pats, progs[s])
        yv = getter(s) if s in sc_i.index else np.nan
        if got is None or not np.isfinite(got[-1]): miss += 1; continue
        if not np.isfinite(yv): miss += 1; continue
        # 对 "E, nu = a, b" 只取需要的那一个
        cv = got[0]
        if np.isclose(cv, yv, rtol=1e-6): ok += 1
        else: bad += 1; mism.append((s, name, cv, yv))
    res.append(dict(field=name, match=ok, mismatch=bad, not_extracted=miss, mismatch_rate=bad / max(1, ok + bad)))
display(pd.DataFrame(res).style.format({"mismatch_rate": "{:.1%}"}))
if mism: display(pd.DataFrame(mism, columns=["scene", "field", "code_value", "yaml_value"]).head(20))

# --- "定义了但没用"启发式：名字只出现一次 => 只有定义没有使用 ---
dead = {}
for name in ["rho", "E", "nu", "mu_0", "lam_0", "mu", "lam", "friction", "mu_f", "yield_stress", "hardening", "viscosity"]:
    defined = used = dead_n = 0
    for c in progs.values():
        n = len(re.findall(rf"\b{name}\b", c))
        if re.search(rf"(^|[\s,]){name}\s*(,[^=\n]*)?=", c, re.M):
            defined += 1
            if n <= 1: dead_n += 1
    if defined: dead[name] = dict(programs_defining=defined, defined_but_unused=dead_n, share_unused=dead_n / defined)
display(pd.DataFrame(dead).T)
# >>> 看什么：
#   1) mismatch_rate>0 => yaml 不是代码的忠实摘要，要决定谁是 ground truth（建议：以代码为准，重新从代码解析 yaml）
#   2) 液体程序里 mu_0 被定义但没用 => yaml 里的 youngs_modulus/poissons_ratio 对液体只通过 lam 起作用，E 与 ν 不可分辨

## 7. 视频检查

In [ ]:
try:
    import cv2
except ImportError:
    cv2 = None; print("cv2 not installed: pip install opencv-python-headless")

if CHECK_VIDEO and cv2 is not None:
    rng = random.Random(0)
    sample = rng.sample(list(specs), min(VIDEO_SAMPLE, len(specs)))
    vrows = []
    for sid in sample:
        vp = DATA_ROOT / sid / VIDEO_NAME
        if not vp.exists(): continue
        cap = cv2.VideoCapture(str(vp))
        vrows.append(dict(scene=sid, n_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), fps=cap.get(cv2.CAP_PROP_FPS),
                          w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
        cap.release()
    vd = pd.DataFrame(vrows)
    if len(vd):
        vd = vd.merge(sc[["scene", "frames", "fps"]].rename(columns={"frames": "yaml_frames", "fps": "yaml_fps"}), on="scene", how="left")
        print("resolution:", vd.groupby(["w", "h"]).size().to_dict())
        print("fps:", vd.fps.round(2).value_counts().to_dict())
        print("n_frames:", vd.n_frames.describe().round(1).to_dict())
        print(f"n_frames == yaml duration_frames: {(vd.n_frames == vd.yaml_frames).mean():.1%} | fps == yaml fps: {np.isclose(vd.fps, vd.yaml_fps).mean():.1%}")
    else:
        print("sampled scenes have no video files")

## 8. 划分分析：泄漏诊断与分组划分

- 若设置了 `SPLIT_FILES`：诊断你现有划分的泄漏程度
- 同时给出两种替代划分：**结构分组划分**、**参数外推划分**，并与随机划分对比泄漏

In [ ]:
meta = scene_df[["scene", "sig"]].merge(ps[["scene", "tmpl", "nn_sim", "nn_scene", "code_tokens"]], on="scene", how="left")
prim = first_dyn[["cm", "rho", "E", "nu"]].reset_index()
meta = meta.merge(prim, on="scene", how="left")
meta["logE"], meta["logrho"] = np.log10(meta.E), np.log10(meta.rho)
meta["logE_over_rho"] = meta.logE - meta.logrho

def load_split_files(files):
    a = {}
    for k, f in files.items():
        for line in Path(f).read_text().split():
            a[line.strip()] = k
    names = {Path(s).name: s for s in meta.scene}
    m = {}
    for key, k in a.items():
        sid = key if key in set(meta.scene) else names.get(Path(key).name)
        if sid: m[sid] = k
    return meta.scene.map(m)

def grouped_split(meta, group_col, test_frac=.1, val_frac=.1, seed=0):
    rng = np.random.default_rng(seed)
    keys = meta[group_col].fillna("__none__")
    groups = keys.value_counts()
    n = len(meta); target = {"test": test_frac * n, "val": val_frac * n}; cur = {"test": 0, "val": 0}; assign = {}
    for g in rng.permutation(groups.index.to_numpy()):
        sz = groups[g]
        for k in ("test", "val"):
            if cur[k] < target[k] and cur[k] + sz <= target[k] * 1.2:
                assign[g] = k; cur[k] += sz; break
        else:
            assign[g] = "train"
    return keys.map(assign)

def random_split(meta, test_frac=.1, val_frac=.1, seed=0):
    rng = np.random.default_rng(seed); p = rng.permutation(len(meta)); n = len(meta)
    lab = np.array(["train"] * n, dtype=object)
    lab[p[:int(test_frac * n)]] = "test"; lab[p[int(test_frac * n):int((test_frac + val_frac) * n)]] = "val"
    return pd.Series(lab, index=meta.index)

def param_ood_split(meta, q=.9, seed=0):
    # 每种材料内：logE/rho 最高 (1-q) 的样本划入 ood_test，其余随机划分
    lab = random_split(meta, seed=seed).copy()
    for g, sub in meta.groupby("cm"):
        thr = sub.logE_over_rho.quantile(q)
        lab[sub.index[sub.logE_over_rho > thr]] = "ood_test"
    return lab

def leakage_report(meta, labels, name):
    tr, te = meta[labels == "train"], meta[labels.isin(["test", "ood_test"])]
    if tr.empty or te.empty: print(f"[{name}] empty train or test"); return
    trs = dict(sizes=labels.value_counts().to_dict(),
        test_tmpl_in_train=te.tmpl.isin(set(tr.tmpl)).mean(),
        test_sig_in_train=te.sig.isin(set(tr.sig)).mean(),
        test_nn_sim_to_train_mean=None)
    # 测试程序到训练程序的最近邻相似度（复用 X）
    ids_idx = {s: i for i, s in enumerate(ids)}
    tr_i = [ids_idx[s] for s in tr.scene if s in ids_idx]; te_i = [ids_idx[s] for s in te.scene if s in ids_idx]
    sims = []
    for s in range(0, len(te_i), 500):
        S = (X[te_i[s:s+500]] @ X[tr_i].T).toarray(); sims.append(S.max(1))
    sims = np.concatenate(sims) if sims else np.array([])
    trs["test_nn_sim_to_train_mean"] = float(sims.mean()) if len(sims) else np.nan
    trs["test_nn_sim_gt_0.95"] = float((sims > .95).mean()) if len(sims) else np.nan
    print(f"[{name}]"); print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in trs.items()}, indent=1, default=str))

cands = {"random": random_split(meta), "grouped_by_sig": grouped_split(meta, "sig"), "grouped_by_template": grouped_split(meta, "tmpl"), "param_ood(logE/rho top10%)": param_ood_split(meta)}
if SPLIT_FILES: cands = {"YOURS": load_split_files(SPLIT_FILES), **cands}
for k, v in cands.items(): leakage_report(meta, v, k)
# >>> 看什么：random 的 test_tmpl_in_train / test_nn_sim_gt_0.95 越高，随机划分越不能测泛化；
#     grouped 版本降到多少，就是"真正的新结构"比例。param_ood 用来测参数外推

In [ ]:
# 划分前后的参数分布对比（KS 检验 + 材料占比）
from scipy.stats import ks_2samp
def compare(labels, name):
    tr = meta[labels == "train"]; te = meta[labels.isin(["test", "ood_test"])]
    rows = []
    for p in ["logrho", "logE", "nu", "logE_over_rho"]:
        a, b = tr[p].dropna(), te[p].dropna()
        if len(a) > 5 and len(b) > 5:
            r = ks_2samp(a, b); rows.append(dict(split=name, param=p, ks=r.statistic, p=r.pvalue))
    return pd.DataFrame(rows)
display(pd.concat([compare(v, k) for k, v in cands.items()]).round(4))
for k, v in cands.items():
    print(f"\n{k}: material share train vs test")
    display(pd.concat({"train": meta[v == "train"].cm.value_counts(normalize=True),
                       "test": meta[v.isin(["test", "ood_test"])].cm.value_counts(normalize=True)}, axis=1).round(3))

## 9. 视频 token 预算（辅助计算）

In [ ]:
# Qwen2.5-VL: patch14 + 2x2 merge => 28px/token，时间 patch=2。Qwen3-VL 的 patch/merge 设置不同（约 32px/token），请以 processor 配置为准
def vid_tokens(T, H, W, unit=28, tpatch=2):
    h = max(unit, round(H / unit) * unit); w = max(unit, round(W / unit) * unit)
    return (T // tpatch) * (h // unit) * (w // unit)

rows = []
for T in (8, 16, 32):
    for (H, W) in [(224, 224), (336, 336), (448, 448), (512, 512)]:
        rows.append(dict(frames=T, res=f"{H}x{W}", qwen25vl_tokens=vid_tokens(T, H, W, 28), qwen3vl_tokens_approx=vid_tokens(T, H, W, 32)))
bud = pd.DataFrame(rows); display(bud)
if "code_tokens" in ps:
    p99 = int(ps.code_tokens.quantile(.99)); print(f"code tokens P99 ≈ {p99}；yaml tokens P99 ≈ {int(ps.yaml_tokens.quantile(.99))}；总长度 = 视频 token + prompt + yaml + code")

## 10. 保存

In [ ]:
meta.to_csv(OUT_DIR / "scene_meta.csv", index=False)
obj_df.to_csv(OUT_DIR / "objects.csv", index=False)
ps.to_csv(OUT_DIR / "program_stats.csv", index=False)
cands_df = pd.DataFrame({"scene": meta.scene, **{k: v for k, v in cands.items()}})
cands_df.to_csv(OUT_DIR / "candidate_splits.csv", index=False)
print("saved to", OUT_DIR.resolve())